# FinBERT Fine-tuning for GECS Industry Classification

Fine-tunes `yiyanghkust/finbert-pretrain` (financial-domain BERT) on 145 GECS industry codes.

**Runtime:** Make sure you select **Runtime → Change runtime type → GPU (T4 or A100)**.

**Inputs you'll upload:**
- `task1_train.csv`
- `task1_test.csv`

**Outputs to download:**
- `finbert_model.zip` — fine-tuned classification model
- `finbert_test_predictions.csv` — predictions on test set with probabilities
- `finbert_embeddings_train.npy` + `finbert_embeddings_test.npy` — for stacking
- `finbert_results.json` — F1 / accuracy / top-10 metrics

In [ ]:
# ── 1. Setup ─────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

!pip install -q transformers==4.44.2 sentence-transformers==3.0.1 accelerate==0.33.0 scikit-learn==1.4.2

In [ ]:
# ── 2. Upload train + test CSVs ──────────────────────────────────────────
from google.colab import files
print('Upload task1_train.csv and task1_test.csv (multi-select):')
uploaded = files.upload()
import pandas as pd
train = pd.read_csv('task1_train.csv')
test  = pd.read_csv('task1_test.csv')
print(f'train: {len(train):,}  test: {len(test):,}')
print(f'classes: {train.mstar_code.nunique()}')

In [ ]:
# ── 3. Build label encoder + tokenize ───────────────────────────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(train['mstar_code'].tolist() + test['mstar_code'].tolist())
train['label'] = le.transform(train['mstar_code'])
test['label']  = le.transform(test['mstar_code'])
n_classes = len(le.classes_)
print(f'n_classes: {n_classes}')

from transformers import AutoTokenizer
MODEL_NAME = 'yiyanghkust/finbert-pretrain'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

def tokenize(texts):
    return tok(texts.tolist(), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')

tr_enc = tokenize(train['text'])
te_enc = tokenize(test['text'])
print('train shape:', tr_enc['input_ids'].shape, '  test shape:', te_enc['input_ids'].shape)

In [ ]:
# ── 4. Model + data loaders ─────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_classes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

import numpy as np
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight('balanced', classes=np.arange(n_classes), y=train['label'].values)
weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
print('class weight stats:', float(weights_t.min()), float(weights_t.max()), float(weights_t.mean()))

BATCH = 32
tr_ds = TensorDataset(tr_enc['input_ids'], tr_enc['attention_mask'], torch.tensor(train['label'].values))
te_ds = TensorDataset(te_enc['input_ids'], te_enc['attention_mask'], torch.tensor(test['label'].values))
tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True)
te_loader = DataLoader(te_ds, batch_size=BATCH * 2, shuffle=False)

In [ ]:
# ── 5. Training loop ────────────────────────────────────────────────────
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

EPOCHS = 3
LR = 2e-5

criterion = nn.CrossEntropyLoss(weight=weights_t)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
n_steps = len(tr_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(n_steps * 0.1), num_training_steps=n_steps)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, (ids, mask, lbl) in enumerate(tqdm(tr_loader, desc=f'epoch {epoch+1}/{EPOCHS}')):
        ids, mask, lbl = ids.to(device), mask.to(device), lbl.to(device)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask)
        loss = criterion(out.logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f'epoch {epoch+1} avg loss: {total_loss/len(tr_loader):.4f}')

In [ ]:
# ── 6. Eval on test set ─────────────────────────────────────────────────
from sklearn.metrics import f1_score, accuracy_score
from collections import Counter

model.eval()
all_preds, all_probs, all_true = [], [], []
with torch.no_grad():
    for ids, mask, lbl in tqdm(te_loader, desc='eval'):
        ids, mask = ids.to(device), mask.to(device)
        out = model(input_ids=ids, attention_mask=mask)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=-1)
        all_preds.extend(preds.tolist())
        all_probs.append(probs)
        all_true.extend(lbl.numpy().tolist())

all_probs = np.vstack(all_probs)
true_codes = le.inverse_transform(all_true)
pred_codes = le.inverse_transform(all_preds)

macro_f1 = f1_score(true_codes, pred_codes, average='macro', zero_division=0)
acc = accuracy_score(true_codes, pred_codes)
cf = Counter(true_codes.tolist())
top10 = [c for c, _ in cf.most_common(10)]
f1s = f1_score(true_codes, pred_codes, average=None, labels=top10, zero_division=0)
n_pass = int(sum(1 for v in f1s if v > 0.85))

print(f'\n FINBERT FINE-TUNED RESULT ')
print(f'Macro F1   : {macro_f1*100:.2f}%')
print(f'Accuracy   : {acc*100:.2f}%')
print(f'Top-10 pass: {n_pass}/10')
for c, v in zip(top10, f1s):
    flag = 'PASS' if v > 0.85 else 'FAIL'
    print(f'  [{flag}] {c}: F1={v*100:.1f}%')

In [ ]:
# ── 7. Extract sentence embeddings (CLS token) for stacking ─────────────
# These can be downloaded and combined with V13/V14 features locally.
from transformers import AutoModel
embedding_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
# Load fine-tuned weights into the encoder body
embedding_model.load_state_dict(
    {k.replace('bert.', ''): v for k, v in model.state_dict().items() if k.startswith('bert.')},
    strict=False,
)
embedding_model.eval()

def encode_dataset(loader):
    embs = []
    with torch.no_grad():
        for ids, mask, _ in tqdm(loader, desc='embed'):
            ids, mask = ids.to(device), mask.to(device)
            out = embedding_model(input_ids=ids, attention_mask=mask)
            cls = out.last_hidden_state[:, 0, :]  # [CLS] token
            embs.append(cls.cpu().numpy())
    return np.vstack(embs)

tr_loader_seq = DataLoader(tr_ds, batch_size=BATCH * 2, shuffle=False)
E_tr = encode_dataset(tr_loader_seq)
E_te = encode_dataset(te_loader)
print('train emb:', E_tr.shape, ' test emb:', E_te.shape)

In [ ]:
# ── 8. Save artifacts and download ─────────────────────────────────────
import json
import os

os.makedirs('finbert_outputs', exist_ok=True)

# Predictions CSV
pd.DataFrame({
    'true_code': true_codes,
    'pred_code': pred_codes,
}).to_csv('finbert_outputs/finbert_test_predictions.csv', index=False)

# Probability matrix (n_test x n_classes)
np.save('finbert_outputs/finbert_test_probs.npy', all_probs)

# Sentence embeddings for stacking
np.save('finbert_outputs/finbert_embeddings_train.npy', E_tr)
np.save('finbert_outputs/finbert_embeddings_test.npy',  E_te)

# Class encoder mapping (so we can decode probs locally)
np.save('finbert_outputs/le_classes.npy', le.classes_)

# Summary metrics
summary = {
    'model': MODEL_NAME,
    'epochs': EPOCHS,
    'batch': BATCH,
    'max_len': MAX_LEN,
    'macro_f1': round(float(macro_f1) * 100, 2),
    'accuracy': round(float(acc) * 100, 2),
    'top10_pass': int(n_pass),
    'n_classes': int(n_classes),
}
with open('finbert_outputs/finbert_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Save model itself
model.save_pretrained('finbert_outputs/finbert_model')
tok.save_pretrained('finbert_outputs/finbert_model')

# Zip and download
!zip -qr finbert_outputs.zip finbert_outputs
from google.colab import files
files.download('finbert_outputs.zip')
print('Downloaded finbert_outputs.zip — unzip into your project folder.')